In [7]:
!pip install numpy>=1.26.0 pandas>=2.2.2 pmdarima>=2.0.3

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.ar_model import AutoReg
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import warnings
warnings.filterwarnings("ignore")

# List to track generated plots and model results
generated_plots = []
model_results = []

# Load Excel files
one_min_files = {
    'TSLA': '1 min TSLA.xlsx',
    'MSFT': '1 min MSFT.xlsx',
    'APPLE': '1 min AAPL.xlsx'
}
thirty_min_files = {
    'TSLA': '30 min TSLA.xlsx',
    'MSFT': '30 min MSFT.xlsx',
    'APPLE': '30 min AAPL.xlsx'
}

one_min_data = {}
thirty_min_data = {}

for stock in ['TSLA', 'MSFT', 'APPLE']:
    one_min_data[stock] = pd.read_excel(one_min_files[stock])
    one_min_data[stock].index = pd.date_range(start='2025-04-28 09:30:00', periods=len(one_min_data[stock]), freq='1min')
    one_min_data[stock] = one_min_data[stock].fillna(method='ffill')

    thirty_min_data[stock] = pd.read_excel(thirty_min_files[stock])
    thirty_min_data[stock].index = pd.date_range(start='2025-04-28 09:30:00', periods=len(thirty_min_data[stock]), freq='30min')
    thirty_min_data[stock] = thirty_min_data[stock].fillna(method='ffill')

# Clean series
def clean_series(series, name):
    print(f"\nChecking {name}:")
    nan_count = series.isna().sum()
    print(f" - NaN values: {nan_count}")
    series = pd.to_numeric(series, errors='coerce').dropna()
    print(f" - Length after cleaning: {len(series)}")
    return series

# Extract and clean closing prices
cleaned_data = {}
for stock in ['TSLA', 'MSFT', 'APPLE']:
    cleaned_data[f'{stock}_1min'] = clean_series(one_min_data[stock][f'{stock} 1 day'], f'{stock} 1-min')
    cleaned_data[f'{stock}_30min'] = clean_series(thirty_min_data[stock][f'{stock} 5 day'], f'{stock} 30-min')

# Ensure consistent length
min_length_1min = min(len(cleaned_data[f'{stock}_1min']) for stock in ['TSLA', 'MSFT', 'APPLE'])
min_length_30min = min(len(cleaned_data[f'{stock}_30min']) for stock in ['TSLA', 'MSFT', 'APPLE'])
for stock in ['TSLA', 'MSFT', 'APPLE']:
    cleaned_data[f'{stock}_1min'] = cleaned_data[f'{stock}_1min'][:min_length_1min]
    cleaned_data[f'{stock}_30min'] = cleaned_data[f'{stock}_30min'][:min_length_30min]

# Stationarity tests
def stationarity_tests(series, name):
    print(f"\nStationarity Tests for {name}:")
    try:
        adf_result = adfuller(series)
        print(f"ADF: Statistic={adf_result[0]:.4f}, p-value={adf_result[1]:.4f}")
        print(f"  - {'Stationary' if adf_result[1] < 0.05 else 'Non-stationary'}")

        kpss_result = kpss(series)
        print(f"KPSS: Statistic={kpss_result[0]:.4f}, p-value={kpss_result[1]:.4f}")
        print(f"  - {'Non-stationary' if kpss_result[1] < 0.05 else 'Stationary'}")
    except Exception as e:
        print(f"Error: {e}")

for stock in ['TSLA', 'MSFT', 'APPLE']:
    stationarity_tests(cleaned_data[f'{stock}_1min'], f'{stock} 1-min')
    stationarity_tests(cleaned_data[f'{stock}_30min'], f'{stock} 30-min')

# Cointegration test
def cointegration_test(series1, series2, name1, name2):
    print(f"\nCointegration Test: {name1} vs {name2}:")
    try:
        series1 = (series1 - series1.mean()) / series1.std()
        series2 = (series2 - series2.mean()) / series2.std()
        data = pd.DataFrame({name1: series1, name2: series2})
        result = coint_johansen(data, det_order=0, k_ar_diff=1)
        print(f"Trace Statistic: {result.lr1}")
        print(f"Critical Values (95%): {result.cvt[:, 1]}")
        for i, stat in enumerate(result.lr1):
            print(f"  - Rank {i}: {'Cointegrated' if stat > result.cvt[i, 1] else 'Not cointegrated'}")
    except Exception as e:
        print(f"  - Error: {e}")

for sheet in ['1min', '30min']:
    suffix = '1-min' if sheet == '1min' else '30-min'
    for pair in [('APPLE', 'MSFT'), ('APPLE', 'TSLA'), ('MSFT', 'TSLA')]:
        cointegration_test(cleaned_data[f'{pair[0]}_{sheet}'], cleaned_data[f'{pair[1]}_{sheet}'],
                          f'{pair[0]} {suffix}', f'{pair[1]} {suffix}')

# Fit AR, ARMA, ARIMA models
def fit_models(series, name, forecast_steps=10):
    print(f"\nModels for {name}:")
    result = {'Stock': name, 'Model': '', 'AIC': '', 'BIC': '', 'Parameters': ''}

    # AR Model
    try:
        ar_model = AutoReg(series, lags=1).fit()
        ar_forecast = ar_model.predict(start=len(series), end=len(series) + forecast_steps - 1)

        plt.figure(figsize=(10, 6))
        plt.plot(series, label='Observed')
        plt.plot(range(len(series), len(series) + forecast_steps), ar_forecast, label='Forecast', color='red')
        plt.title(f'AR Forecast for {name}')
        plt.legend()
        ar_plot = f'ar_forecast_{name.replace(" ", "_")}.png'
        plt.savefig(ar_plot)
        plt.close()
        if os.path.exists(ar_plot):
            generated_plots.append(ar_plot)
            print(f"Saved: {ar_plot}")

        result['Model'] = 'AR(1)'
        result['AIC'] = f"{ar_model.aic:.2f}"
        result['BIC'] = f"{ar_model.bic:.2f}"
        result['Parameters'] = f"AR1: {ar_model.params[1]:.3f}"
        model_results.append(result.copy())
    except Exception as e:
        print(f"AR Error: {e}")

    # ARMA Model
    try:
        arma_model = auto_arima(series, seasonal=False, max_p=2, max_q=2, suppress_warnings=True)
        arma_forecast = arma_model.predict(n_periods=forecast_steps)

        plt.figure(figsize=(10, 6))
        plt.plot(series, label='Observed')
        plt.plot(range(len(series), len(series) + forecast_steps), arma_forecast, label='Forecast', color='red')
        plt.title(f'ARMA Forecast for {name}')
        plt.legend()
        arma_plot = f'arma_forecast_{name.replace(" ", "_")}.png'
        plt.savefig(arma_plot)
        plt.close()
        if os.path.exists(arma_plot):
            generated_plots.append(arma_plot)
            print(f"Saved: {arma_plot}")

        result['Model'] = f'ARMA{arma_model.order}'
        result['AIC'] = f"{arma_model.aic():.2f}"
        result['BIC'] = f"{arma_model.bic():.2f}"
        params = arma_model.params()
        param_str = ', '.join([f"{k}: {v:.3f}" for k, v in zip(arma_model.param_names(), params)])
        result['Parameters'] = param_str
        model_results.append(result.copy())
    except Exception as e:
        print(f"ARMA Error: {e}")

    # ARIMA Model
    try:
        arima_model = ARIMA(series, order=(1, 1, 1)).fit()
        arima_forecast = arima_model.forecast(steps=forecast_steps)

        plt.figure(figsize=(10, 6))
        plt.plot(series, label='Observed')
        plt.plot(range(len(series), len(series) + forecast_steps), arima_forecast, label='Forecast', color='red')
        plt.title(f'ARIMA Forecast for {name}')
        plt.legend()
        arima_plot = f'arima_forecast_{name.replace(" ", "_")}.png'
        plt.savefig(arima_plot)
        plt.close()
        if os.path.exists(arima_plot):
            generated_plots.append(arima_plot)
            print(f"Saved: {arima_plot}")

        residuals = arima_model.resid
        plt.figure(figsize=(10, 4))
        plt.subplot(121)
        plot_acf(residuals, ax=plt.gca(), title='ACF')
        plt.subplot(122)
        plot_pacf(residuals, ax=plt.gca(), title='PACF')
        plt.tight_layout()
        resid_plot = f'residuals_{name.replace(" ", "_")}.png'
        plt.savefig(resid_plot)
        plt.close()
        if os.path.exists(resid_plot):
            generated_plots.append(resid_plot)
            print(f"Saved: {resid_plot}")

        result['Model'] = 'ARIMA(1,1,1)'
        result['AIC'] = f"{arima_model.aic:.2f}"
        result['BIC'] = f"{arima_model.bic:.2f}"
        params = arima_model.params
        param_str = ', '.join([f"{k}: {v:.3f}" for k, v in zip(arima_model.param_names, params)])
        result['Parameters'] = param_str
        model_results.append(result.copy())
    except Exception as e:
        print(f"ARIMA Error: {e}")

# Apply models to 30-min data
for stock in ['TSLA', 'MSFT', 'APPLE']:
    fit_models(cleaned_data[f'{stock}_30min'], f'{stock} 30-min')

# Plot all series
plt.figure(figsize=(12, 8))
for stock in ['TSLA', 'MSFT', 'APPLE']:
    plt.plot(cleaned_data[f'{stock}_1min'], label=f'{stock} 1-min')
    plt.plot(cleaned_data[f'{stock}_30min'], label=f'{stock} 30-min')
plt.title('Closing Prices')
plt.legend()
comparison_plot = 'closing_prices_comparison.png'
plt.savefig(comparison_plot)
plt.close()
if os.path.exists(comparison_plot):
    generated_plots.append(comparison_plot)
    print(f"Saved: {comparison_plot}")

# Generate model results table
results_df = pd.DataFrame(model_results)
table_md = results_df.to_markdown(index=False)

# Generate report
report = f"""
# Time Series Report
## Stationarity
- [ADF/KPSS results]
## Cointegration
- [Johansen results]
## Models
- AR, ARMA, ARIMA fitted.
### Model Results
{table_md}
### Plots
{chr(10).join([f'- {plot}' for plot in generated_plots])}
"""

with open('report.md', 'w') as f:
    f.write(report)



Checking TSLA 1-min:
 - NaN values: 0
 - Length after cleaning: 563

Checking TSLA 30-min:
 - NaN values: 0
 - Length after cleaning: 159

Checking MSFT 1-min:
 - NaN values: 0
 - Length after cleaning: 563

Checking MSFT 30-min:
 - NaN values: 0
 - Length after cleaning: 159

Checking APPLE 1-min:
 - NaN values: 0
 - Length after cleaning: 563

Checking APPLE 30-min:
 - NaN values: 0
 - Length after cleaning: 159

Stationarity Tests for TSLA 1-min:
ADF: Statistic=0.0313, p-value=0.9610
  - Non-stationary
KPSS: Statistic=0.6895, p-value=0.0145
  - Non-stationary

Stationarity Tests for TSLA 30-min:
ADF: Statistic=-3.2301, p-value=0.0183
  - Stationary
KPSS: Statistic=0.1416, p-value=0.1000
  - Stationary

Stationarity Tests for MSFT 1-min:
ADF: Statistic=-2.8709, p-value=0.0488
  - Stationary
KPSS: Statistic=1.5244, p-value=0.0100
  - Non-stationary

Stationarity Tests for MSFT 30-min:
ADF: Statistic=-1.1716, p-value=0.6857
  - Non-stationary
KPSS: Statistic=1.6001, p-value=0.0100
  -

In [8]:
def fit_models(series, name, forecast_steps=10):
    print(f"\nModels for {name}:")
    result = {'Stock': name, 'Model': '', 'AIC': '', 'BIC': '', 'Parameters': ''}

    # AR Model
    try:
        ar_model = AutoReg(series, lags=1).fit()
        ar_forecast = ar_model.predict(start=len(series), end=len(series) + forecast_steps - 1)

        plt.figure(figsize=(10, 6))
        plt.plot(series, label='Observed')
        plt.plot(range(len(series), len(series) + forecast_steps), ar_forecast, label='Forecast', color='red')
        plt.title(f'AR Forecast for {name}')
        plt.legend()
        ar_plot = f'ar_forecast_{name.replace(" ", "_")}.png'
        plt.savefig(ar_plot)
        plt.close()
        if os.path.exists(ar_plot):
            generated_plots.append(ar_plot)
            print(f"Saved: {ar_plot}")

        result['Model'] = 'AR(1)'
        result['AIC'] = f"{ar_model.aic:.2f}"
        result['BIC'] = f"{ar_model.bic:.2f}"
        result['Parameters'] = f"AR1: {ar_model.params[1]:.3f}"
        model_results.append(result.copy())
    except Exception as e:
        print(f"AR Error: {e}")

    # ARMA Model
    try:
        arma_model = auto_arima(series, seasonal=False, max_p=2, max_q=2, suppress_warnings=True)
        arma_forecast = arma_model.predict(n_periods=forecast_steps)

        plt.figure(figsize=(10, 6))
        plt.plot(series, label='Observed')
        plt.plot(range(len(series), len(series) + forecast_steps), arma_forecast, label='Forecast', color='red')
        plt.title(f'ARMA Forecast for {name}')
        plt.legend()
        arma_plot = f'arma_forecast_{name.replace(" ", "_")}.png'
        plt.savefig(arma_plot)
        plt.close()
        if os.path.exists(arma_plot):
            generated_plots.append(arma_plot)  # Add ARMA plot to generated_plots
            print(f"Saved: {arma_plot}")

        result['Model'] = f'ARMA{arma_model.order}'
        result['AIC'] = f"{arma_model.aic():.2f}"
        result['BIC'] = f"{arma_model.bic():.2f}"
        params = arma_model.params()
        param_str = ', '.join([f"{k}: {v:.3f}" for k, v in zip(arma_model.param_names(), params)])
        result['Parameters'] = param_str
        model_results.append(result.copy())
    except Exception as e:
        print(f"ARMA Error: {e}")

    # ARIMA Model
    try:
        arima_model = ARIMA(series, order=(1, 1, 1)).fit()
        arima_forecast = arima_model.forecast(steps=forecast_steps)

        plt.figure(figsize=(10, 6))
        plt.plot(series, label='Observed')
        plt.plot(range(len(series), len(series) + forecast_steps), arima_forecast, label='Forecast', color='red')
        plt.title(f'ARIMA Forecast for {name}')
        plt.legend()
        arima_plot = f'arima_forecast_{name.replace(" ", "_")}.png'
        plt.savefig(arima_plot)
        plt.close()
        if os.path.exists(arima_plot):
            generated_plots.append(arima_plot)  # Add ARIMA plot to generated_plots
            print(f"Saved: {arima_plot}")

        residuals = arima_model.resid
        plt.figure(figsize=(10, 4))
        plt.subplot(121)
        plot_acf(residuals, ax=plt.gca(), title='ACF')
        plt.subplot(122)
        plot_pacf(residuals, ax=plt.gca(), title='PACF')
        plt.tight_layout()
        resid_plot = f'residuals_{name.replace(" ", "_")}.png'
        plt.savefig(resid_plot)
        plt.close()
        if os.path.exists(resid_plot):
            generated_plots.append(resid_plot)
            print(f"Saved: {resid_plot}")

        result['Model'] = 'ARIMA(1,1,1)'
        result['AIC'] = f"{arima_model.aic:.2f}"
        result['BIC'] = f"{arima_model.bic:.2f}"
        params = arima_model.params
        param_str = ', '.join([f"{k}: {v:.3f}" for k, v in zip(arima_model.param_names, params)])
        result['Parameters'] = param_str
        model_results.append(result.copy())
    except Exception as e:
        print(f"ARIMA Error: {e}")

# Apply models to 30-min data
for stock in ['TSLA', 'MSFT', 'APPLE']:
    fit_models(cleaned_data[f'{stock}_30min'], f'{stock} 30-min')

# Generate model results table
results_df = pd.DataFrame(model_results)
table_md = results_df.to_markdown(index=False)

# Generate report with ARMA and ARIMA plots included
report = f"""
# Time Series Report
## Stationarity
- [ADF/KPSS results]
## Cointegration
- [Johansen results]
## Models
- AR, ARMA, ARIMA fitted.
### Model Results
{table_md}
### Plots
{chr(10).join([f'- {plot}' for plot in generated_plots])}
"""

with open('report.md', 'w') as f:
    f.write(report)



Models for TSLA 30-min:
Saved: ar_forecast_TSLA_30-min.png
Saved: arma_forecast_TSLA_30-min.png
ARMA Error: 'ARIMA' object has no attribute 'param_names'
Saved: arima_forecast_TSLA_30-min.png
Saved: residuals_TSLA_30-min.png

Models for MSFT 30-min:
Saved: ar_forecast_MSFT_30-min.png
Saved: arma_forecast_MSFT_30-min.png
ARMA Error: 'ARIMA' object has no attribute 'param_names'
Saved: arima_forecast_MSFT_30-min.png
Saved: residuals_MSFT_30-min.png

Models for APPLE 30-min:
Saved: ar_forecast_APPLE_30-min.png
Saved: arma_forecast_APPLE_30-min.png
ARMA Error: 'ARIMA' object has no attribute 'param_names'
Saved: arima_forecast_APPLE_30-min.png
Saved: residuals_APPLE_30-min.png
